## Import thư viện

In [4]:
import random
from collections import deque

## Input - Random trạng thái ban đầu

In [5]:
def input_floor():
    m = int(input("Nhập số dòng: "))
    n = int(input("Nhập số cột: "))
    
    floor = [] #Danh sách dùng để lưu ma trận đầu vào
    
    for i in range(m):
        row = []
        for j in range(n):
            row.append(random.choice([0,1])) # Random giá trị: 0 là sàn sạch, 1 là sàn bẩn
        floor.append(row) 
             
    #Khởi tạo vị trí ban đầu của máy hút bụi
    pos = (random.randint(0, m-1), random.randint(0, n-1))
    vx, vy = pos
    
    floor[vx][vy]= "V"
    
    return floor

## Output

In [6]:
def output_floor(floor):
    for row in floor:
        print(row)

## Các hàm xử lý ma trận

### Copy ma trận

In [7]:
#Copy để để khi xử lý tránh ảnh hưởng ma trận gốc
def copy_floor(floor):
    return [row[:] for row in floor]

### Kiểm tra trạng thái đích

In [8]:
def goal(floor):
    for row in floor:
        for cell in row:
            if cell == 1: return False
    return True  

### Tìm vị trí máy hút bụi

In [9]:
def find_vacuum(floor):
    for i in range(len(floor)):
        for j in range(len(floor[0])):
            if floor[i][j] == "V": return (i,j)

### Lấy các hành động có thể đi của máy hút bụi tại một thời điểm

In [10]:
def get_possible_moves(floor):
    vx, vy = find_vacuum(floor)
    
    m = len(floor)
    n = len(floor[0])
    
    moves = []
    
    #Kiểm tra các biên
    if vx > 0: moves.append("UP")
    if vx < m-1: moves.append("DOWN")
    if vy > 0: moves.append("LEFT")
    if vy < n-1: moves.append("RIGHT")
    
    return moves

### Tình vị trí mới của máy hút bụi sau khi hành động

In [11]:
def apply_move(pos, action):
    vx, vy = pos
    
    if action == "UP": vx -= 1
    if action == "DOWN": vx += 1
    if action == "LEFT": vy -= 1
    if action == "RIGHT": vy += 1
    
    return (vx, vy)

### Hàm sinh ra trạng thái mới của ma trận sau mỗi hành động

In [12]:
def move_vacuum(floor, action):
    new_floor = copy_floor(floor)
    
    old_vx, old_vy = find_vacuum(floor)
    new_vx, new_vy = apply_move((old_vx, old_vy), action)
    
    new_floor[old_vx][old_vy] = 0
    new_floor[new_vx][new_vy] = "V"
    
    return new_floor
    

### Chuyển ma trận thành key để lưu vào reached

In [13]:
def state_key(floor):
    return tuple(tuple(row) for row in floor)

## Thuật toán bfs

In [14]:
def bfs(initial_floor):
    node = {
        "state" : initial_floor, #Lưu trạng thái hiện tại của ma trận
        "path" : [initial_floor] #Lưu danh sách các bước đi từ trạng thái ban đầu đến hiện tại
    }
        
    if goal(node["state"]): return node
    
    frontier = deque()
    frontier.append(node)
    
    reached = set()
    reached.add(state_key(initial_floor))
    
    while len(frontier) > 0:
        node = frontier.popleft()
                
        for action in get_possible_moves(node["state"]):
            child_state = move_vacuum(node["state"], action)
            
            child = {
                "state": child_state,
                "path": node["path"] + [child_state]
            }
            if state_key(child_state) not in reached:
                
                if goal(child_state): return child
                
                reached.add(state_key(child_state))
                frontier.append(child)
    return None    

## Hàm Main

In [15]:
floor = input_floor()

print("\nTrạng thái ban đầu của sàn nhà: ")
output_floor(floor)

result = bfs(floor)
if result is None: print("\nKhông tìm thấy lời giải!!!")
else:
    print("\nĐường đi BFS tìm được:")
    
    for i in range(len(result["path"])):
        print(f"\nTrạng thái {i}:")
        output_floor(result["path"][i])
        
    print("\nSố bước đi ngắn nhất:", len(result["path"]) - 1)
    
    print("\nTrạng thái cuối cùng của ma trận: ")
    output_floor(result["state"])


Trạng thái ban đầu của sàn nhà: 
[1, 1, 0, 1]
['V', 1, 1, 1]
[1, 1, 1, 1]
[1, 1, 1, 0]

Đường đi BFS tìm được:

Trạng thái 0:
[1, 1, 0, 1]
['V', 1, 1, 1]
[1, 1, 1, 1]
[1, 1, 1, 0]

Trạng thái 1:
['V', 1, 0, 1]
[0, 1, 1, 1]
[1, 1, 1, 1]
[1, 1, 1, 0]

Trạng thái 2:
[0, 'V', 0, 1]
[0, 1, 1, 1]
[1, 1, 1, 1]
[1, 1, 1, 0]

Trạng thái 3:
[0, 0, 0, 1]
[0, 'V', 1, 1]
[1, 1, 1, 1]
[1, 1, 1, 0]

Trạng thái 4:
[0, 0, 0, 1]
[0, 0, 1, 1]
[1, 'V', 1, 1]
[1, 1, 1, 0]

Trạng thái 5:
[0, 0, 0, 1]
[0, 0, 1, 1]
['V', 0, 1, 1]
[1, 1, 1, 0]

Trạng thái 6:
[0, 0, 0, 1]
[0, 0, 1, 1]
[0, 0, 1, 1]
['V', 1, 1, 0]

Trạng thái 7:
[0, 0, 0, 1]
[0, 0, 1, 1]
[0, 0, 1, 1]
[0, 'V', 1, 0]

Trạng thái 8:
[0, 0, 0, 1]
[0, 0, 1, 1]
[0, 0, 1, 1]
[0, 0, 'V', 0]

Trạng thái 9:
[0, 0, 0, 1]
[0, 0, 1, 1]
[0, 0, 'V', 1]
[0, 0, 0, 0]

Trạng thái 10:
[0, 0, 0, 1]
[0, 0, 'V', 1]
[0, 0, 0, 1]
[0, 0, 0, 0]

Trạng thái 11:
[0, 0, 'V', 1]
[0, 0, 0, 1]
[0, 0, 0, 1]
[0, 0, 0, 0]

Trạng thái 12:
[0, 0, 0, 'V']
[0, 0, 0, 1]
[0, 0, 0, 1]
[